# 01 — Bronze → Silver

Reads raw orders from **Snowflake** (`RAPPI_SANDBOX.SYNTH.ORDER_DIMENSIONS_SYNTH`) and lands a cleaned, partitioned **Delta** silver table in the AIDP standard catalog.

**Cluster requirements**
- `spark-snowflake_2.12-3.1.1.jar` + `snowflake-jdbc-3.19.0.jar` on the classpath (provided under `assets/aidp/jars/`).
- Delta is enabled by default on AIDP Spark runtimes.

**What this notebook does**
1. Reads from Snowflake with **column projection** + **optional date watermark** pushed down to Snowflake (less network bytes).
2. Applies silver-grade cleaning: drop rows missing identity, normalize casing, derive `order_date` / `order_hour` / `revenue_net` / `is_delivered` / `is_cancelled`, stamp lineage.
3. Writes Delta partitioned by `country_code, order_date` — same shape Snowflake clusters bronze on, so downstream gold reads stay narrow.


## Parameters

Non-sensitive params (Snowflake host/db, silver target) come from `assets/aidp/config.yaml`. Snowflake `sfUser`/`sfPassword` come from the AIDP credential named under `snowflake.credential`. Set `CONFIG_PATH` + `AIDP_ENV` at the top of the next cell.

`INCREMENTAL_FROM` stays as an env-var runtime toggle — set it (`YYYY-MM-DD`) for incremental loads, leave unset for full refresh.


In [ ]:
# ══════════════════ Notebook config ══════════════════════════════════
# CONFIG_PATH : workspace path to assets/aidp/config.yaml. Get it from
#               the AIDP UI: right-click config.yaml → 'Copy path'.
# AIDP_ENV    : which block of config.yaml to load ('dev' or 'prod').
#               Defaults to 'dev'; Airflow can override via env var.
# ─────────────────────────────────────────────────────────────────────
import os

CONFIG_PATH = '/Workspace/sales-orders/assets/aidp/config.yaml'
AIDP_ENV    = os.environ.get('AIDP_ENV', 'dev')
print('env: ', AIDP_ENV)
# ═════════════════════════════════════════════════════════════════════

import time
from pyspark.sql import functions as F, types as T


# Incremental watermark — runtime toggle, NOT in config.yaml. Set
# as a notebook param / Airflow env var to switch this run to
# incremental mode. None = full refresh.
INCREMENTAL_FROM = os.environ.get('INCREMENTAL_FROM')   # 'YYYY-MM-DD' or None


In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    _cfg_all = yaml.safe_load(f)

if AIDP_ENV not in _cfg_all:
    raise KeyError(f'env "{AIDP_ENV}" not in {CONFIG_PATH}; available: {sorted(_cfg_all)}')

cfg = _cfg_all[AIDP_ENV]
TEST_ROWS = cfg.get('test_rows', {}).get('bronze', 0)
print(f'env    : {AIDP_ENV}')
print(f'config : {CONFIG_PATH}')

# Snowflake: merge non-sensitive options from config + user/password from AIDP secret.
_snow_cfg     = cfg['snowflake']
_cred_name    = _snow_cfg['credential']
snow          = {k: v for k, v in _snow_cfg.items() if k != 'credential'}
snow['sfUser']     = aidputils.secrets.get(name=_cred_name, key='user')
snow['sfPassword'] = aidputils.secrets.get(name=_cred_name, key='password')
print(f'snow   : {snow["sfDatabase"]}.{snow["sfSchema"]} @ {snow["sfUrl"]} (cred={_cred_name})')

SOURCE_TABLE = cfg['source']['table']
DEST_CATALOG = cfg['silver']['catalog']
DEST_SCHEMA  = cfg['silver']['schema']
DEST_TABLE   = cfg['silver']['table']
DEST_FQN     = '.'.join(p for p in [DEST_CATALOG, DEST_SCHEMA, DEST_TABLE] if p)

print(f'source : snowflake://{snow["sfDatabase"]}.{snow["sfSchema"]}.{SOURCE_TABLE}')
print(f'target : {DEST_FQN} (delta, partitioned by country_code, order_date)')
print(f'mode   : {"incremental from " + INCREMENTAL_FROM if INCREMENTAL_FROM else "full refresh"}')
if TEST_ROWS:
    print(f'TEST_ROWS={TEST_ROWS} — limited pull')


## 1. Read from Snowflake with pushdown

Two pushdown moves matter here:

- **Projection pushdown** — we explicitly list columns in `query`. The connector pushes the `SELECT` list to Snowflake so only those columns are arrow-encoded and shipped.
- **Predicate pushdown** — when `INCREMENTAL_FROM` is set, the `WHERE ORDER_CREATED_AT >= ...` lands in Snowflake. Combined with the `CLUSTER BY (TO_DATE(ORDER_CREATED_AT), COUNTRY_CODE)` on bronze, Snowflake prunes micro-partitions before any bytes leave the warehouse.

The connector parallelizes the read automatically via Snowflake's internal stage + arrow result chunks — no need for `partitionColumn`/`numPartitions` like generic JDBC. Tune `partition_size_in_mb` only if you see executors starved or skewed.


In [ ]:
where = f"WHERE ORDER_CREATED_AT >= TO_TIMESTAMP_NTZ('{INCREMENTAL_FROM}')" if INCREMENTAL_FROM else ''
limit = f'LIMIT {TEST_ROWS}' if TEST_ROWS else ''

query = f"""
SELECT
  ORDER_ID, USER_ID, STORE_ID, COURIER_ID,
  ORDER_CREATED_AT, ORDER_DELIVERED_AT, ORDER_CANCELLED_AT, DELIVERY_TIME_MINUTES,
  COUNTRY_CODE, CITY, LATITUDE, LONGITUDE,
  VERTICAL,
  TOTAL_AMOUNT, PRODUCT_AMOUNT, DELIVERY_FEE, SERVICE_FEE, TIP, DISCOUNT,
  CURRENCY, PAYMENT_METHOD,
  USER_SEGMENT, ORDER_NUMBER_FOR_USER, ACQUISITION_CHANNEL,
  IS_PRIME_USER, PRIME_TIER,
  FRAUD_SCORE, IS_FRAUDULENT, FRAUD_RULE_TRIGGERED,
  ORDER_STATUS, CANCELLATION_REASON, DELIVERY_TYPE, IS_RECURRENT_ORDER
FROM {SOURCE_TABLE}
{where}
{limit}
"""

bronze = (
    spark.read.format('snowflake')
        .options(**snow)
        .option('query', query)
        # Bigger chunks = fewer Spark partitions = less scheduling overhead at this size.
        # Default 8MB is fine; bump to 64MB for very large pulls.
        .option('partition_size_in_mb', '64')
        .load()
)

print(f'bronze partitions: {bronze.rdd.getNumPartitions()}')
bronze.printSchema()


## 2. Silver transformations

Cheap, deterministic, all expressed in the DataFrame API so Catalyst can fuse them into one stage. No `collect()`, no Python UDFs.


In [ ]:
silver = (
    bronze
    # Identity / status are NOT NULL contracts for silver downstream.
    .filter(F.col('ORDER_ID').isNotNull() & F.col('COUNTRY_CODE').isNotNull() & F.col('ORDER_STATUS').isNotNull())
    .withColumnRenamed('ORDER_ID',                'order_id')
    .withColumnRenamed('USER_ID',                 'user_id')
    .withColumnRenamed('STORE_ID',                'store_id')
    .withColumnRenamed('COURIER_ID',              'courier_id')
    .withColumnRenamed('ORDER_CREATED_AT',        'order_created_at')
    .withColumnRenamed('ORDER_DELIVERED_AT',      'order_delivered_at')
    .withColumnRenamed('ORDER_CANCELLED_AT',      'order_cancelled_at')
    .withColumnRenamed('DELIVERY_TIME_MINUTES',   'delivery_time_minutes')
    .withColumnRenamed('COUNTRY_CODE',            'country_code')
    .withColumnRenamed('CITY',                    'city')
    .withColumnRenamed('LATITUDE',                'latitude')
    .withColumnRenamed('LONGITUDE',               'longitude')
    # NOTE: Spark is case-INSENSITIVE by default — `withColumn('foo', F.lower(F.col('FOO')))`
    # already replaces FOO in-place. Don't `.drop('FOO')` afterward; it drops the new column.
    .withColumn      ('vertical',                 F.lower(F.col('VERTICAL')))
    .withColumnRenamed('TOTAL_AMOUNT',            'total_amount')
    .withColumnRenamed('PRODUCT_AMOUNT',          'product_amount')
    .withColumnRenamed('DELIVERY_FEE',            'delivery_fee')
    .withColumnRenamed('SERVICE_FEE',             'service_fee')
    .withColumnRenamed('TIP',                     'tip')
    .withColumnRenamed('DISCOUNT',                'discount')
    .withColumn      ('currency',                 F.upper(F.col('CURRENCY')))
    .withColumn      ('payment_method',           F.lower(F.col('PAYMENT_METHOD')))
    .withColumn      ('user_segment',             F.lower(F.col('USER_SEGMENT')))
    .withColumnRenamed('ORDER_NUMBER_FOR_USER',   'order_number_for_user')
    .withColumn      ('acquisition_channel',      F.lower(F.col('ACQUISITION_CHANNEL')))
    .withColumnRenamed('IS_PRIME_USER',           'is_prime_user')
    .withColumn      ('prime_tier',               F.lower(F.col('PRIME_TIER')))
    .withColumnRenamed('FRAUD_SCORE',             'fraud_score')
    .withColumnRenamed('IS_FRAUDULENT',           'is_fraudulent')
    .withColumn      ('fraud_rule_triggered',     F.lower(F.col('FRAUD_RULE_TRIGGERED')))
    .withColumn      ('order_status',             F.lower(F.col('ORDER_STATUS')))
    .withColumn      ('cancellation_reason',      F.lower(F.col('CANCELLATION_REASON')))
    .withColumn      ('delivery_type',            F.lower(F.col('DELIVERY_TYPE')))
    .withColumnRenamed('IS_RECURRENT_ORDER',      'is_recurrent_order')
    # --- Derived columns (denormalize what every downstream mart will recompute otherwise) ---
    .withColumn('order_date',   F.to_date('order_created_at'))                       # partition col
    .withColumn('order_hour',   F.hour('order_created_at').cast(T.ByteType()))
    .withColumn('revenue_net',  (F.col('total_amount') - F.coalesce(F.col('discount'), F.lit(0))).cast(T.DecimalType(18, 2)))
    .withColumn('is_delivered', F.col('order_status') == F.lit('delivered'))
    .withColumn('is_cancelled', F.col('order_status') == F.lit('cancelled'))
    .withColumn('bronze_ingested_at', F.current_timestamp())                          # lineage
)

silver.printSchema()


## 3. Write to Delta silver

- `partitionBy('country_code', 'order_date')` — matches the predicate shape gold marts will use.
- For **full refresh** we use `mode('overwrite')`.
- For **incremental** runs (`INCREMENTAL_FROM` set) we use Delta's `replaceWhere` to atomically replace just the affected partitions — no full table rewrite, no readers see torn state.
- Repartition before write so each Spark task writes one file per `(country_code, order_date)` partition instead of many small ones.


In [ ]:
# `mode('overwrite').saveAsTable()` is Spark's canonical "create if not
# exists, else replace" pattern: first run materializes the table,
# subsequent runs atomically overwrite the data (and partitions).
spark.sql(f'CREATE SCHEMA IF NOT EXISTS ' + '.'.join(p for p in [DEST_CATALOG, DEST_SCHEMA] if p))

writer = (
    silver.repartition('country_code', 'order_date')
          .write.format('delta')
          .partitionBy('country_code', 'order_date')
          .option('overwriteSchema', 'true')
)

if INCREMENTAL_FROM:
    (writer
        .mode('overwrite')
        .option('replaceWhere', f"order_date >= DATE'{INCREMENTAL_FROM}'")
        .saveAsTable(DEST_FQN))
else:
    writer.mode('overwrite').saveAsTable(DEST_FQN)

print(f'wrote: {DEST_FQN}')


## 4. Sanity check


In [ ]:
rows = spark.table(DEST_FQN).count()
by_country = (spark.table(DEST_FQN)
                   .groupBy('country_code').count()
                   .orderBy(F.col('count').desc()))
by_country.show(truncate=False)

print(f'silver_rows={rows}')
print(f'silver_table={DEST_FQN}')
